In [ ]:
import scanpy as sc
import numpy as np
import muon as mu

In [ ]:
adata = sc.read_h5ad("./Data/RNA_ATAC_ADT/TEA-seq/TEA-seq.h5ad")
adata_gex = adata[:, adata.var['modality'] == "Gene Expression"]
adata_atac = adata[:, adata.var['modality'] == "Peaks"]
print(adata_gex)
print(adata_atac)

View of AnnData object with n_obs × n_vars = 25517 × 36601
    obs: 'cell_type', 'batch'
    var: 'modality'
    obsm: 'protein_expression'
View of AnnData object with n_obs × n_vars = 25517 × 128853
    obs: 'cell_type', 'batch'
    var: 'modality'
    obsm: 'protein_expression'


In [ ]:
sc.pp.normalize_total(adata_gex, target_sum=1e4)
sc.pp.log1p(adata_gex)
sc.pp.highly_variable_genes(adata_gex, n_top_genes=4000, batch_key='batch', subset=True)

sc.pp.normalize_total(adata_atac)
sc.pp.log1p(adata_atac)
sc.pp.highly_variable_genes(adata_atac, n_top_genes=10000, batch_key='batch', flavor="cell_ranger", subset=True)

adata_adt = sc.AnnData(X = adata.obsm["protein_expression"].astype(float))
adata_adt.obs = adata.obs
mu.prot.pp.clr(adata_adt)
adata_adt.layers['clr']=np.matrix(adata_adt.X.copy())

In [4]:
mdata = mu.MuData({'rna': adata_gex, 'atac': adata_atac, 'adt': adata_adt})
print(mdata)

MuData object with n_obs × n_vars = 25517 × 14046
  3 modalities
    rna:	25517 x 4000
      obs:	'cell_type', 'batch'
      var:	'modality', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
      uns:	'log1p', 'hvg'
      obsm:	'protein_expression'
    atac:	25517 x 10000
      obs:	'cell_type', 'batch'
      var:	'modality', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
      uns:	'log1p', 'hvg'
      obsm:	'protein_expression'
    adt:	25517 x 46
      obs:	'cell_type', 'batch'
      layers:	'clr'


In [ ]:
mu.tl.mofa(mdata, groups_label='batch', gpu_mode=False)


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
       
 
        
Loaded view='rna' group='6' with N=6194 samples and D=4000 features...
Loaded view='rna' group='4' with N=6381 samples and D=4000 features...
Loaded view='rna' group='5' with N=6412 samples and D=4000 features...
Loaded view='rna' group='3' with N=6530 samples and D=4000 features...
Loaded view='atac' group='6' with N=6194 samples and D=10000 features...
Loaded view='atac' group

In [ ]:
np.save('MOFA_tea_3.npy', mdata.obsm['X_mofa'])